In [ ]:
# Péndulo Doble — Simulación y Animación

**Modelo:** Doble péndulo plano, dos masas puntuales m1, m2 unidas por varillas rígidas sin masa de longitudes L1, L2.  
**Variables de estado:** θ1, θ2 (ángulos desde la vertical, en radianes), ω1=dθ1/dt, ω2=dθ2/dt (rad/s).  
**Unidades:** kg, m, s, rad. Gravedad g en m/s².  
**Amortiguamiento:** b1, b2 (N·m·s/rad). Si b=0 el sistema es conservativo (energía constante).  
**Caos:** Pequeñas diferencias en CI generan trayectorias completamente distintas en pocos segundos.

**Cómo ajustar parámetros:**  
- Edita el diccionario `PARAMS` al inicio de la celda de código.  
- Para ver caos, activa `compare_chaos=True` y ajusta `delta_theta` (perturbación en θ1).  
- Para guardar la animación, cambia `save_as` a `"pendulum.mp4"` o `"pendulum.gif"`.  
- Aumenta `t_max` para simulaciones más largas; reduce `dt` para mayor precisión visual.
</VSCode.Cell>
<VSCode.Cell language="python">
# =============================================================================
# PÉNDULO DOBLE — script completo
# Modelo, integración numérica, animación y análisis de sensibilidad al caos.
# =============================================================================
"""
Doble péndulo plano con amortiguamiento viscoso opcional.

Ecuaciones de movimiento (Lagrangiano, ver comentarios en `_equations`):
  Las ecuaciones son no lineales y acopladas. Se resuelven con solve_ivp (DOP853).

Unidades SI: kg, m, s, rad, m/s².
Para correr: ejecuta esta celda. Ajusta PARAMS según necesites.
"""

from __future__ import annotations

import sys
import warnings
from dataclasses import dataclass, field
from typing import Dict, Any, Optional

import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec

# ── Detectar entorno notebook ────────────────────────────────────────────────
def _in_notebook() -> bool:
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False

if _in_notebook():
    matplotlib.rcParams["animation.html"] = "jshtml"
    # %matplotlib inline  ← descomenta si usas jupyter clásico sin widget

# =============================================================================
# PARÁMETROS — edita aquí
# =============================================================================
PARAMS: Dict[str, Any] = dict(
    # Propiedades físicas
    m1        = 1.0,      # masa bob 1 [kg]
    m2        = 1.0,      # masa bob 2 [kg]
    L1        = 1.0,      # longitud varilla 1 [m]
    L2        = 1.0,      # longitud varilla 2 [m]
    g         = 9.81,     # gravedad [m/s²]
    b1        = 0.0,      # amortiguamiento sobre ω1 [N·m·s/rad]
    b2        = 0.0,      # amortiguamiento sobre ω2 [N·m·s/rad]

    # Condiciones iniciales [rad, rad, rad/s, rad/s]
    theta1_0  = np.pi * 0.6,
    theta2_0  = np.pi * 0.5,
    omega1_0  = 0.0,
    omega2_0  = 0.0,

    # Simulación
    t_max       = 20.0,   # tiempo total [s]
    dt          = 0.02,   # paso de muestreo [s]

    # Animación
    trail_length = 200,   # puntos en el rastro del bob 2 (int) o segundos (float <t_max)
    interval     = 20,    # ms entre frames de la animación

    # Caos: comparar dos trayectorias casi iguales
    compare_chaos = True,
    delta_theta   = 1e-3, # perturbación en θ1 de la segunda trayectoria [rad]

    # Guardar animación: None | "pendulum.mp4" | "pendulum.gif"
    save_as = None,
)

# =============================================================================
# MODELO FÍSICO
# =============================================================================

@dataclass
class SimResults:
    """Resultados de la simulación."""
    t:      np.ndarray
    theta1: np.ndarray
    theta2: np.ndarray
    omega1: np.ndarray
    omega2: np.ndarray
    energy: np.ndarray
    params: Dict[str, Any]
    # Segunda trayectoria (caos), opcional
    theta1b: Optional[np.ndarray] = field(default=None)
    theta2b: Optional[np.ndarray] = field(default=None)


def _equations(t: float, state: np.ndarray, m1: float, m2: float,
               L1: float, L2: float, g: float,
               b1: float, b2: float) -> list[float]:
    """
    Ecuaciones de movimiento del doble péndulo (forma explícita).

    Estado: [θ1, θ2, ω1, ω2]

    Lagrangiano → ecuaciones de Euler-Lagrange:
    ┌                                                                 ┐
    │ (m1+m2)L1²·α1 + m2·L1·L2·cos(δ)·α2                           │
    │     = -m2·L1·L2·ω2²·sin(δ) - (m1+m2)·g·L1·sin(θ1) - b1·ω1  │
    │                                                                 │
    │ m2·L2²·α2 + m2·L1·L2·cos(δ)·α1                               │
    │     =  m2·L1·L2·ω1²·sin(δ) - m2·g·L2·sin(θ2) - b2·ω2        │
    └                                                                 ┘
    donde δ = θ1 - θ2, α1 = dω1/dt, α2 = dω2/dt.

    Resolvemos el sistema 2×2 para α1, α2 usando la regla de Cramer.
    """
    th1, th2, w1, w2 = state
    d = th1 - th2                    # diferencia de ángulos
    sin_d, cos_d = np.sin(d), np.cos(d)
    sin1, sin2   = np.sin(th1), np.sin(th2)

    # Denominador común (determinante de la matriz de masa)
    denom = (m1 + m2) * L1**2 * (m2 * L2**2) - (m2 * L1 * L2 * cos_d)**2

    # Términos del lado derecho
    rhs1 = (-m2 * L1 * L2 * w2**2 * sin_d
            - (m1 + m2) * g * L1 * sin1
            - b1 * w1)
    rhs2 = (m2 * L1 * L2 * w1**2 * sin_d
            - m2 * g * L2 * sin2
            - b2 * w2)

    # Regla de Cramer
    alpha1 = (m2 * L2**2 * rhs1 - m2 * L1 * L2 * cos_d * rhs2) / denom
    alpha2 = ((m1 + m2) * L1**2 * rhs2 - m2 * L1 * L2 * cos_d * rhs1) / denom

    return [w1, w2, alpha1, alpha2]


def _compute_energy(theta1: np.ndarray, theta2: np.ndarray,
                    omega1: np.ndarray, omega2: np.ndarray,
                    m1: float, m2: float, L1: float, L2: float,
                    g: float) -> np.ndarray:
    """Energía mecánica total E = T + V (con V=0 en el pivote)."""
    # Posiciones
    x1 = L1 * np.sin(theta1)
    y1 = -L1 * np.cos(theta1)
    x2 = x1 + L2 * np.sin(theta2)
    y2 = y1 - L2 * np.cos(theta2)

    # Energía cinética
    vx1 = L1 * omega1 * np.cos(theta1)
    vy1 = L1 * omega1 * np.sin(theta1)
    vx2 = vx1 + L2 * omega2 * np.cos(theta2)
    vy2 = vy1 + L2 * omega2 * np.sin(theta2)
    T = 0.5 * m1 * (vx1**2 + vy1**2) + 0.5 * m2 * (vx2**2 + vy2**2)

    # Energía potencial (referencia: pivote)
    V = m1 * g * y1 + m2 * g * y2

    return T + V


def _integrate(state0: list[float], p: Dict[str, Any]) -> np.ndarray:
    """Integra las ecuaciones y devuelve array (4, N)."""
    t_eval = np.arange(0, p["t_max"] + p["dt"], p["dt"])
    sol = solve_ivp(
        fun=_equations,
        t_span=(0.0, p["t_max"]),
        y0=state0,
        method="DOP853",
        t_eval=t_eval,
        args=(p["m1"], p["m2"], p["L1"], p["L2"], p["g"], p["b1"], p["b2"]),
        rtol=1e-10, atol=1e-12,
        dense_output=False,
    )
    return sol


def simulate(params: Dict[str, Any]) -> SimResults:
    """
    Integra el doble péndulo con los parámetros dados.

    Parameters
    ----------
    params : dict  — ver PARAMS al inicio del script.

    Returns
    -------
    SimResults
    """
    p = params
    state0 = [p["theta1_0"], p["theta2_0"], p["omega1_0"], p["omega2_0"]]

    print("Integrando trayectoria principal…", end=" ", flush=True)
    sol = _integrate(state0, p)
    print("OK" if sol.success else f"¡ADVERTENCIA! {sol.message}")

    # Chequeo de estabilidad
    if not np.all(np.isfinite(sol.y)):
        raise ValueError("La solución contiene NaN/Inf. "
                         "Reduce dt o revisa los parámetros.")

    th1, th2, w1, w2 = sol.y
    energy = _compute_energy(th1, th2, w1, w2,
                             p["m1"], p["m2"], p["L1"], p["L2"], p["g"])

    # Reporte de energía
    if p["b1"] == 0 and p["b2"] == 0:
        e_err = np.abs(energy - energy[0]) / np.abs(energy[0])
        print(f"  Error relativo de energía máx: {e_err.max():.2e}")
    else:
        print(f"  Energía inicial: {energy[0]:.4f} J  |  final: {energy[-1]:.4f} J")

    # Segunda trayectoria para análisis de caos
    th1b = th2b = None
    if p.get("compare_chaos"):
        state0b = state0.copy()
        state0b[0] += p["delta_theta"]
        print("Integrando trayectoria perturbada…", end=" ", flush=True)
        solb = _integrate(state0b, p)
        print("OK")
        if not np.all(np.isfinite(solb.y)):
            warnings.warn("Trayectoria perturbada explota. Se omite comparación.")
        else:
            th1b, th2b = solb.y[0], solb.y[1]
            # Recortar al mismo tamaño si difieren
            n = min(len(th1), len(th1b))
            th1, th2, w1, w2, energy = (arr[:n] for arr in (th1, th2, w1, w2, energy))
            th1b, th2b = th1b[:n], th2b[:n]
            sol.t = sol.t[:n]

    return SimResults(
        t=sol.t, theta1=th1, theta2=th2,
        omega1=w1, omega2=w2,
        energy=energy, params=p,
        theta1b=th1b, theta2b=th2b,
    )

# =============================================================================
# VISUALIZACIÓN: gráfica de energía
# =============================================================================

def plot_energy(res: SimResults) -> None:
    """Grafica la energía total en función del tiempo."""
    p = res.params
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(res.t, res.energy, color="steelblue", lw=1.2, label="E total")
    ax.set_xlabel("Tiempo [s]")
    ax.set_ylabel("Energía [J]")
    title = "Energía mecánica total"
    if p["b1"] == 0 and p["b2"] == 0:
        e_err = np.abs(res.energy - res.energy[0]) / np.abs(res.energy[0])
        title += f"  (error rel. máx = {e_err.max():.2e})"
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# =============================================================================
# ANIMACIÓN
# =============================================================================

def _bob_positions(theta1: np.ndarray, theta2: np.ndarray,
                   L1: float, L2: float) -> tuple:
    """Calcula (x1, y1, x2, y2) a partir de los ángulos."""
    x1 = L1 * np.sin(theta1)
    y1 = -L1 * np.cos(theta1)
    x2 = x1 + L2 * np.sin(theta2)
    y2 = y1 - L2 * np.cos(theta2)
    return x1, y1, x2, y2


def animate(res: SimResults, params: Dict[str, Any]) -> animation.FuncAnimation:
    """
    Crea y muestra la animación del doble péndulo.

    Parameters
    ----------
    res    : SimResults — salida de simulate().
    params : dict       — mismo diccionario PARAMS.

    Returns
    -------
    FuncAnimation (útil para mostrar en notebook).
    """
    p = params
    has_chaos = res.theta1b is not None

    L1, L2 = p["L1"], p["L2"]
    lim = (L1 + L2) * 1.15

    # Trail: convertir segundos → puntos
    trail = p["trail_length"]
    if isinstance(trail, float) and trail < p["t_max"]:
        dt = p["dt"]
        trail = max(1, int(trail / dt))
    trail = int(trail)

    x1, y1, x2, y2 = _bob_positions(res.theta1, res.theta2, L1, L2)
    if has_chaos:
        x1b, y1b, x2b, y2b = _bob_positions(res.theta1b, res.theta2b, L1, L2)

    # ── Layout ───────────────────────────────────────────────────────────────
    if has_chaos:
        fig = plt.figure(figsize=(10, 6))
        gs  = GridSpec(2, 2, figure=fig,
                       width_ratios=[3, 1], height_ratios=[3, 1])
        ax_pend  = fig.add_subplot(gs[:, 0])  # animación principal
        ax_chaos = fig.add_subplot(gs[1, 1])  # distancia angular
        ax_pend2 = None
    else:
        fig, ax_pend = plt.subplots(figsize=(6, 6))
        ax_chaos = None

    # ── Ejes del péndulo ─────────────────────────────────────────────────────
    ax_pend.set_xlim(-lim, lim)
    ax_pend.set_ylim(-lim, lim)
    ax_pend.set_aspect("equal")
    ax_pend.set_facecolor("#0d0d0d")
    ax_pend.grid(True, color="#333333", lw=0.5)
    ax_pend.set_title("Péndulo Doble", color="white", pad=8)
    fig.patch.set_facecolor("#1a1a1a")
    for sp in ax_pend.spines.values():
        sp.set_color("#444")
    ax_pend.tick_params(colors="#888")

    # Elementos gráficos — trayectoria A
    rod_a,   = ax_pend.plot([], [], lw=2.5, color="#4fc3f7", solid_capstyle="round")
    bob1_a,  = ax_pend.plot([], [], "o", ms=12, color="#4fc3f7", zorder=5)
    bob2_a,  = ax_pend.plot([], [], "o", ms=10, color="#ff8a65", zorder=5)
    trail_a, = ax_pend.plot([], [], lw=1, color="#ff8a65", alpha=0.5, zorder=4)
    pivot_a, = ax_pend.plot([0], [0], "o", ms=6, color="white", zorder=6)

    # Elementos gráficos — trayectoria B (caos)
    if has_chaos:
        rod_b,   = ax_pend.plot([], [], lw=2.5, color="#a5d6a7",
                                solid_capstyle="round", alpha=0.75)
        bob1_b,  = ax_pend.plot([], [], "o", ms=12, color="#a5d6a7",
                                zorder=5, alpha=0.75)
        bob2_b,  = ax_pend.plot([], [], "o", ms=10, color="#fff176",
                                zorder=5, alpha=0.75)
        trail_b, = ax_pend.plot([], [], lw=1, color="#fff176",
                                alpha=0.4, zorder=4)

        # Subplot de distancia angular
        ax_chaos.set_facecolor("#0d0d0d")
        ax_chaos.tick_params(colors="#888", labelsize=7)
        ax_chaos.set_title("|Δθ1|+|Δθ2|", color="white", fontsize=8, pad=4)
        ax_chaos.set_xlabel("t [s]", color="#888", fontsize=7)
        for sp in ax_chaos.spines.values():
            sp.set_color("#444")
        chaos_line, = ax_chaos.plot([], [], lw=1, color="#ff6090")
        dist_all = np.abs(res.theta1 - res.theta1b) + np.abs(res.theta2 - res.theta2b)
        ax_chaos.set_xlim(0, res.t[-1])
        ax_chaos.set_ylim(0, dist_all.max() * 1.1 + 1e-10)

    time_text = ax_pend.text(0.02, 0.96, "", transform=ax_pend.transAxes,
                             color="white", fontsize=10, va="top")

    # ── Función de actualización ──────────────────────────────────────────────
    def _update(frame: int):
        # --- trayectoria A ---
        px = [0, x1[frame], x2[frame]]
        py = [0, y1[frame], y2[frame]]
        rod_a.set_data(px, py)
        bob1_a.set_data([x1[frame]], [y1[frame]])
        bob2_a.set_data([x2[frame]], [y2[frame]])
        start = max(0, frame - trail)
        trail_a.set_data(x2[start:frame+1], y2[start:frame+1])

        time_text.set_text(f"t = {res.t[frame]:.2f} s")

        artists = [rod_a, bob1_a, bob2_a, trail_a, pivot_a, time_text]

        if has_chaos:
            pxb = [0, x1b[frame], x2b[frame]]
            pyb = [0, y1b[frame], y2b[frame]]
            rod_b.set_data(pxb, pyb)
            bob1_b.set_data([x1b[frame]], [y1b[frame]])
            bob2_b.set_data([x2b[frame]], [y2b[frame]])
            trail_b.set_data(x2b[start:frame+1], y2b[start:frame+1])

            chaos_line.set_data(res.t[:frame+1], dist_all[:frame+1])
            artists += [rod_b, bob1_b, bob2_b, trail_b, chaos_line]

        return artists

    n_frames = len(res.t)
    anim = animation.FuncAnimation(
        fig, _update, frames=n_frames,
        interval=p["interval"], blit=True,
    )

    # ── Guardar ───────────────────────────────────────────────────────────────
    save_as = p.get("save_as")
    if save_as:
        ext = save_as.rsplit(".", 1)[-1].lower()
        try:
            if ext == "mp4":
                writer = animation.FFMpegWriter(fps=int(1000/p["interval"]),
                                               bitrate=1800)
            else:
                writer = animation.PillowWriter(fps=int(1000/p["interval"]))
            anim.save(save_as, writer=writer)
            print(f"Animación guardada en '{save_as}'")
        except Exception as e:
            print(f"No se pudo guardar la animación: {e}")
            print("  Para MP4 instala ffmpeg (https://ffmpeg.org).")
            print("  Para GIF instala Pillow: pip install pillow")

    plt.tight_layout()

    if _in_notebook():
        from IPython.display import display, HTML
        plt.close(fig)
        display(HTML(anim.to_jshtml()))
    else:
        plt.show()

    return anim

# =============================================================================
# PUNTO DE ENTRADA
# =============================================================================

if __name__ == "__main__":
    results = simulate(PARAMS)
    plot_energy(results)
    anim = animate(results, PARAMS)

# 🔬 Péndulo Doble — Simulación y Animación

## Modelo Físico

El **péndulo doble** consiste en dos masas puntuales $m_1$, $m_2$ unidas por varillas rígidas sin masa de longitudes $L_1$, $L_2$.

**Variables de estado:** $\theta_1$, $\theta_2$ (ángulos desde la vertical), $\omega_1 = \dot\theta_1$, $\omega_2 = \dot\theta_2$.

**Unidades:** SI — metros [m], kilogramos [kg], segundos [s], radianes [rad].

**Parámetros ajustables** (celda de configuración):
- `m1, m2`: masas [kg]
- `L1, L2`: longitudes [m]
- `g`: gravedad [m/s²]
- `b1, b2`: coeficientes de amortiguamiento viscoso [kg·m²/s] (0 = sin amortiguamiento)
- `theta1_0, theta2_0`: ángulos iniciales [rad]
- `omega1_0, omega2_0`: velocidades angulares iniciales [rad/s]
- `t_max, dt`: tiempo total y paso de muestreo [s]
- `trail_length`: longitud del rastro del segundo bob [número de puntos]
- `COMPARE_CHAOS`: activar comparación de trayectorias caóticas (bool)
- `SAVE_ANIMATION`: guardar como MP4/GIF (bool)

In [ ]:
# ─── Imports ──────────────────────────────────────────────────────────────────
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.integrate import solve_ivp
from typing import Dict, Any, Tuple

In [ ]:
# ─── Parameters ───────────────────────────────────────────────────────────────
PARAMS: Dict[str, Any] = {
    # Physical parameters
    "m1": 1.0,        # mass 1 [kg]
    "m2": 1.0,        # mass 2 [kg]
    "L1": 1.0,        # rod 1 length [m]
    "L2": 1.0,        # rod 2 length [m]
    "g":  9.81,       # gravity [m/s²]
    "b1": 0.0,        # damping on ω1 [kg·m²/s]
    "b2": 0.0,        # damping on ω2 [kg·m²/s]

    # Initial conditions
    "theta1_0": np.pi / 2,    # [rad]
    "theta2_0": np.pi / 2,    # [rad]
    "omega1_0": 0.0,           # [rad/s]
    "omega2_0": 0.0,           # [rad/s]

    # Integration
    "t_max": 20.0,    # total simulation time [s]
    "dt":    0.02,    # sampling step [s]

    # Visualisation
    "trail_length": 150,    # number of trail points for bob 2
    "fps":          50,     # animation frames per second

    # Options
    "COMPARE_CHAOS":  True,    # compare two nearly-identical initial conditions
    "chaos_delta":    1e-5,    # perturbation in theta1 for chaos comparison [rad]
    "SAVE_ANIMATION": False,   # set True to save (requires ffmpeg for mp4)
    "save_filename":  "double_pendulum.mp4",  # or .gif
}

In [ ]:
# ─── Physics model ────────────────────────────────────────────────────────────

def equations_of_motion(
    t: float,
    state: np.ndarray,
    m1: float, m2: float,
    L1: float, L2: float,
    g: float,
    b1: float, b2: float,
) -> np.ndarray:
    """
    Equations of motion for the double pendulum (Lagrangian formulation).

    State vector: [θ1, θ2, ω1, ω2]

    Derived from the Euler-Lagrange equations. The mass matrix M and
    forcing vector F are:

        M = [ (m1+m2)L1²        m2 L1 L2 cos(Δθ) ]
            [ m2 L1 L2 cos(Δθ)  m2 L2²            ]

        F1 = -m2 L1 L2 ω2² sin(Δθ) - (m1+m2) g L1 sin(θ1) - b1 ω1
        F2 = +m2 L1 L2 ω1² sin(Δθ) - m2 g L2 sin(θ2)        - b2 ω2

    where Δθ = θ1 - θ2. Solved via Cramer's rule.
    """
    th1, th2, w1, w2 = state
    dth = th1 - th2
    cos_d = np.cos(dth)
    sin_d = np.sin(dth)

    # Mass matrix elements
    M11 = (m1 + m2) * L1**2
    M12 = m2 * L1 * L2 * cos_d
    M21 = M12
    M22 = m2 * L2**2
    det = M11 * M22 - M12 * M21  # always > 0

    # Forcing vector
    F1 = (-m2 * L1 * L2 * w2**2 * sin_d
          - (m1 + m2) * g * L1 * np.sin(th1)
          - b1 * w1)
    F2 = (m2 * L1 * L2 * w1**2 * sin_d
          - m2 * g * L2 * np.sin(th2)
          - b2 * w2)

    # Cramer's rule: [dω1, dω2] = M⁻¹ F
    alpha1 = (M22 * F1 - M12 * F2) / det
    alpha2 = (M11 * F2 - M21 * F1) / det

    return np.array([w1, w2, alpha1, alpha2])


def total_energy(
    state: np.ndarray,
    m1: float, m2: float,
    L1: float, L2: float,
    g: float,
) -> np.ndarray:
    """
    Compute total mechanical energy E = T + V for each time step.
    V is measured with the pivot as reference (y positive upward).
    """
    th1, th2, w1, w2 = state  # each shape (N,) when called with solution arrays
    # Positions (y positive upward → pivot at origin)
    x1 =  L1 * np.sin(th1)
    y1 = -L1 * np.cos(th1)
    x2 = x1 + L2 * np.sin(th2)
    y2 = y1 - L2 * np.cos(th2)

    # Velocities
    vx1 = L1 * w1 * np.cos(th1)
    vy1 = L1 * w1 * np.sin(th1)
    vx2 = vx1 + L2 * w2 * np.cos(th2)
    vy2 = vy1 + L2 * w2 * np.sin(th2)

    T = 0.5 * m1 * (vx1**2 + vy1**2) + 0.5 * m2 * (vx2**2 + vy2**2)
    V = m1 * g * y1 + m2 * g * y2
    return T + V

In [ ]:
# ─── Simulation ───────────────────────────────────────────────────────────────

def simulate(params: Dict[str, Any]) -> Dict[str, Any]:
    """
    Integrate the double pendulum ODE and return solution arrays.

    Returns a dict with keys:
        t, th1, th2, w1, w2 : solution arrays
        x1, y1, x2, y2      : Cartesian positions
        energy              : total energy array
        params              : input params (pass-through)
    """
    m1, m2  = params["m1"],  params["m2"]
    L1, L2  = params["L1"],  params["L2"]
    g       = params["g"]
    b1, b2  = params["b1"],  params["b2"]
    t_max   = params["t_max"]
    dt      = params["dt"]

    state0 = np.array([
        params["theta1_0"], params["theta2_0"],
        params["omega1_0"], params["omega2_0"],
    ])
    t_eval = np.arange(0.0, t_max + dt, dt)

    sol = solve_ivp(
        equations_of_motion,
        (0.0, t_max),
        state0,
        method="DOP853",
        t_eval=t_eval,
        args=(m1, m2, L1, L2, g, b1, b2),
        rtol=1e-10, atol=1e-12,
        dense_output=False,
    )

    # Stability check
    if not sol.success or np.any(~np.isfinite(sol.y)):
        raise RuntimeError(
            f"Integration failed or diverged: {sol.message}"
        )

    th1, th2, w1, w2 = sol.y

    # Cartesian positions (y upward, pivot at origin)
    x1 =  L1 * np.sin(th1);  y1 = -L1 * np.cos(th1)
    x2 = x1 + L2 * np.sin(th2);  y2 = y1 - L2 * np.cos(th2)

    energy = total_energy(sol.y, m1, m2, L1, L2, g)

    return dict(
        t=sol.t,
        th1=th1, th2=th2, w1=w1, w2=w2,
        x1=x1, y1=y1, x2=x2, y2=y2,
        energy=energy,
        params=params,
    )

In [ ]:
# ─── Energy validation plot ────────────────────────────────────────────────────

def plot_energy(results: Dict[str, Any]) -> None:
    """Plot total energy over time and relative error."""
    E   = results["energy"]
    t   = results["t"]
    E0  = E[0]
    rel = (E - E0) / (np.abs(E0) + 1e-30)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(t, E, color="steelblue", lw=1.2)
    axes[0].set_xlabel("Time [s]")
    axes[0].set_ylabel("E = T + V  [J]")
    axes[0].set_title("Total mechanical energy")
    axes[0].grid(True, alpha=0.4)

    axes[1].plot(t, rel, color="tomato", lw=1.0)
    axes[1].set_xlabel("Time [s]")
    axes[1].set_ylabel("Relative error")
    axes[1].set_title("Energy relative error  $(E-E_0)/|E_0|$")
    axes[1].grid(True, alpha=0.4)

    plt.suptitle(
        f"b1={results['params']['b1']}, b2={results['params']['b2']}  "
        f"(non-zero damping → energy loss expected)",
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Animation ────────────────────────────────────────────────────────────────

def animate(
    results: Dict[str, Any],
    results_b: Dict[str, Any] | None = None,
) -> animation.FuncAnimation:
    """
    Build and return a FuncAnimation object.

    If `results_b` is provided, a second (perturbed) trajectory is shown
    alongside the first and a chaos-distance subplot is added.

    Parameters
    ----------
    results   : output of simulate() for trajectory A
    results_b : output of simulate() for trajectory B (chaos comparison)
    """
    params       = results["params"]
    L1, L2       = params["L1"], params["L2"]
    trail_length = params["trail_length"]
    compare      = results_b is not None
    lim          = (L1 + L2) * 1.15

    # ── Figure layout ────────────────────────────────────────────────────────
    if compare:
        fig = plt.figure(figsize=(10, 6), facecolor="#0d0d0d")
        ax_pend = fig.add_axes([0.03, 0.08, 0.58, 0.88], facecolor="#0d0d0d")
        ax_dist = fig.add_axes([0.68, 0.12, 0.30, 0.35], facecolor="#111")
    else:
        fig = plt.figure(figsize=(6, 6), facecolor="#0d0d0d")
        ax_pend = fig.add_axes([0.05, 0.05, 0.90, 0.90], facecolor="#0d0d0d")

    ax_pend.set_xlim(-lim, lim)
    ax_pend.set_ylim(-lim, lim)
    ax_pend.set_aspect("equal")
    ax_pend.tick_params(colors="white")
    for spine in ax_pend.spines.values():
        spine.set_edgecolor("#444")
    ax_pend.set_facecolor("#0d0d0d")
    ax_pend.set_title("Double Pendulum", color="white", fontsize=13)

    # ── Pendulum A drawing objects ────────────────────────────────────────────
    color_a = "#00d4ff"
    rod_a,   = ax_pend.plot([], [], "-", color=color_a, lw=2.5, zorder=3)
    bob_a,   = ax_pend.plot([], [], "o", color=color_a, ms=10, zorder=4)
    trail_a, = ax_pend.plot([], [], "-", color=color_a, lw=0.8,
                             alpha=0.5, zorder=2)

    # ── Pendulum B (chaos) ────────────────────────────────────────────────────
    if compare:
        color_b = "#ff6b35"
        rod_b,   = ax_pend.plot([], [], "-", color=color_b, lw=2.5, zorder=3)
        bob_b,   = ax_pend.plot([], [], "o", color=color_b, ms=10, zorder=4)
        trail_b, = ax_pend.plot([], [], "-", color=color_b, lw=0.8,
                                 alpha=0.5, zorder=2)

    # Pivot
    ax_pend.plot(0, 0, "o", color="white", ms=6, zorder=5)

    # Time label
    time_text = ax_pend.text(
        -lim * 0.95, lim * 0.90, "", color="white", fontsize=10
    )

    # ── Chaos distance subplot ────────────────────────────────────────────────
    if compare:
        t_full = results["t"]
        dist   = (np.abs(results["th1"] - results_b["th1"]) +
                  np.abs(results["th2"] - results_b["th2"]))

        ax_dist.set_xlim(0, t_full[-1])
        ax_dist.set_ylim(0, max(dist.max() * 1.05, 0.1))
        ax_dist.set_xlabel("t [s]", color="white", fontsize=8)
        ax_dist.set_ylabel("|Δθ₁|+|Δθ₂|", color="white", fontsize=8)
        ax_dist.set_title("Angular distance", color="white", fontsize=9)
        ax_dist.tick_params(colors="white", labelsize=7)
        for spine in ax_dist.spines.values():
            spine.set_edgecolor("#555")
        ax_dist.plot(t_full, dist, color="#aaa", lw=0.6, alpha=0.5)
        dist_line, = ax_dist.plot([], [], color="#ff6b35", lw=1.5)

        # legend
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], color=color_a, lw=2, label="Traj A"),
            Line2D([0], [0], color=color_b, lw=2, label=f"Traj B (Δθ={params['chaos_delta']:.0e})"),
        ]
        ax_pend.legend(handles=legend_elements, loc="upper right",
                       facecolor="#222", labelcolor="white", fontsize=9)

    # ── Frame update function ─────────────────────────────────────────────────
    def update(frame: int):
        # Slice trail
        t0 = max(0, frame - trail_length)

        # Trajectory A
        xa1 = results["x1"][frame]; ya1 = results["y1"][frame]
        xa2 = results["x2"][frame]; ya2 = results["y2"][frame]

        rod_a.set_data([0, xa1, xa2], [0, ya1, ya2])
        bob_a.set_data([0, xa1, xa2], [0, ya1, ya2])
        trail_a.set_data(results["x2"][t0:frame+1],
                         results["y2"][t0:frame+1])

        artists = [rod_a, bob_a, trail_a, time_text]

        time_text.set_text(f"t = {results['t'][frame]:.2f} s")

        if compare:
            xb1 = results_b["x1"][frame]; yb1 = results_b["y1"][frame]
            xb2 = results_b["x2"][frame]; yb2 = results_b["y2"][frame]

            rod_b.set_data([0, xb1, xb2], [0, yb1, yb2])
            bob_b.set_data([0, xb1, xb2], [0, yb1, yb2])
            trail_b.set_data(results_b["x2"][t0:frame+1],
                             results_b["y2"][t0:frame+1])

            dist_line.set_data(results["t"][:frame+1], dist[:frame+1])
            artists += [rod_b, bob_b, trail_b, dist_line]

        return artists

    n_frames = len(results["t"])
    interval = max(1, int(1000 / params["fps"]))

    ani = animation.FuncAnimation(
        fig, update,
        frames=n_frames,
        interval=interval,
        blit=True,
    )
    return ani

In [ ]:
# ─── Save helper ──────────────────────────────────────────────────────────────

def save_animation(ani: animation.FuncAnimation, filename: str) -> None:
    """
    Save animation to MP4 (requires ffmpeg) or GIF (requires Pillow).
    Install ffmpeg: https://ffmpeg.org/download.html
    Install Pillow: pip install Pillow
    """
    ext = filename.split(".")[-1].lower()
    try:
        if ext == "mp4":
            writer = animation.FFMpegWriter(fps=50, bitrate=1800)
            ani.save(filename, writer=writer, dpi=150)
            print(f"✅ Saved MP4: {filename}")
        elif ext == "gif":
            writer = animation.PillowWriter(fps=30)
            ani.save(filename, writer=writer, dpi=100)
            print(f"✅ Saved GIF: {filename}")
        else:
            print(f"⚠️  Unknown extension '{ext}'. Use .mp4 or .gif")
    except Exception as exc:
        print(f"❌ Could not save animation: {exc}")
        print("   → For MP4: install ffmpeg (https://ffmpeg.org)")
        print("   → For GIF: pip install Pillow")

In [ ]:
# ─── Main execution ───────────────────────────────────────────────────────────

# Run simulation A
print("⏳ Simulating trajectory A …")
results_a = simulate(PARAMS)
print(f"✅ Done — {len(results_a['t'])} time steps, "
      f"E₀={results_a['energy'][0]:.4f} J, "
      f"Eₙ={results_a['energy'][-1]:.4f} J")

# Run simulation B (chaos comparison)
results_b = None
if PARAMS["COMPARE_CHAOS"]:
    params_b = PARAMS.copy()
    params_b["theta1_0"] = PARAMS["theta1_0"] + PARAMS["chaos_delta"]
    print("⏳ Simulating trajectory B (perturbed) …")
    results_b = simulate(params_b)
    print("✅ Done")

In [ ]:
# ─── Energy plot ──────────────────────────────────────────────────────────────

plot_energy(results_a)

In [ ]:
# ─── Build and show animation ─────────────────────────────────────────────────

# Enable interactive animation in Jupyter
%matplotlib notebook

ani = animate(results_a, results_b)

if PARAMS["SAVE_ANIMATION"]:
    save_animation(ani, PARAMS["save_filename"])

plt.show()

## 💡 Notas de uso

| Parámetro | Efecto |
|-----------|--------|
| `theta1_0 = theta2_0 ≈ π` | Comportamiento altamente caótico |
| `b1, b2 > 0` | Amortiguamiento — la energía decrece |
| `chaos_delta` | Más pequeño → divergencia tarda más en aparecer |
| `trail_length` | Aumentar para ver más rastro del segundo bob |

### Guardar animación
Cambia `SAVE_ANIMATION: True` y `save_filename` a `"pendulo.mp4"` o `"pendulo.gif"` en la celda de parámetros.

> **ffmpeg** es necesario para MP4: [https://ffmpeg.org/download.html](https://ffmpeg.org/download.html)  
> **Pillow** es necesario para GIF: `pip install Pillow`